## 27. §3.6/Figure 4の「default postureへの収束」記述の訂正(再測定)

Frontiersエディター(Dr. Rahman)への自己申告、およびReviewer 1コメント#5での指摘を受けた、
§3.6・Figure 4の訂正用の再測定セクション。

**発見した問題点**: §13(セル41-43、`run_sensory_rollout_with_full_log`)と、
§26(セル80-81、`run_extended_prefall_log_rollout`)はいずれも、`env.step()`が返った
**直後**に`robot.data.joint_pos`等を読み取っていた。これは「リセット処理が走る前に読んで
いるはず」という想定に基づいていたが、実際には`mjlab`の`ManagerBasedRlEnv.step()`の実装
(`manager_based_rl_env.py`)を確認したところ、終端(`dones=True`)したサブ環境に対する
`self._reset_idx(reset_env_ids)`の呼び出しは、**`step()`の内部で、`step()`がリターンする
前に**実行されていることが判明した:

```python
reset_env_ids = self.reset_buf.nonzero(as_tuple=False).squeeze(-1)
if len(reset_env_ids) > 0:
    self._reset_idx(reset_env_ids)   # ← step()の戻り値を作る前に、既にリセットが走る
    self.scene.write_data_to_sim()
...
return (self.obs_buf, self.reward_buf, self.reset_terminated, self.reset_time_outs, self.extras)
```

つまり、「`env.step()`の戻り値を受け取った直後に読む」という従来の実装では、転倒が
検出されたステップに限り、**物理シミュレーション上の真の転倒姿勢ではなく、リセット後の
デフォルト姿勢を読んでしまっていた**。これが、§3.6で「転倒時の関節角度がdefault postureに
正確に一致する」という(誤った)結果になっていた原因である。

**修正方法**: `raw_env._reset_idx`を一時的にモンキーパッチし、実際のリセット処理が
呼ばれる**直前**に、関節角度・関節速度・PD目標位置・エフォート・ルート速度・足接地状態の
スナップショットを取得してから、元の`_reset_idx`を呼び出す。これにより、物理シミュレー
ション上の真の終端状態を、一切シミュレーションの挙動を変えずに(リセット自体は通常通り
実行されるので、以降のロールアウトにも影響しない)取得できる。

In [ ]:
import contextlib


@contextlib.contextmanager
def capture_true_terminal_state(raw_env, robot, contact_sensor, env_idx=0):
    """raw_env._reset_idx を一時的にフックし、リセットが実行される直前の
    物理状態(env_idxのサブ環境分)をキャプチャする。with節を抜けると元に戻す。

    返り値(dict)のキー:
      captured, joint_pos, joint_vel, joint_pos_target, joint_effort_target,
      root_lin_vel_w, foot_contact_found, foot_contact_force
    """
    capture = {"captured": False}
    original_reset_idx = raw_env._reset_idx

    def capturing_reset_idx(env_ids):
        env_ids_list = env_ids.detach().cpu().tolist() if hasattr(env_ids, "detach") else list(env_ids)
        if env_idx in env_ids_list:
            capture["joint_pos"] = robot.data.joint_pos[env_idx].detach().cpu().numpy().copy()
            capture["joint_vel"] = robot.data.joint_vel[env_idx].detach().cpu().numpy().copy()
            capture["joint_pos_target"] = robot.data.joint_pos_target[env_idx].detach().cpu().numpy().copy()
            capture["joint_effort_target"] = robot.data.joint_effort_target[env_idx].detach().cpu().numpy().copy()
            capture["root_lin_vel_w"] = robot.data.root_link_lin_vel_w[env_idx].detach().cpu().numpy().copy()
            capture["foot_contact_found"] = contact_sensor.data.found[env_idx].detach().cpu().numpy().copy()
            capture["foot_contact_force"] = contact_sensor.data.force[env_idx].detach().cpu().numpy().copy()
            capture["captured"] = True
        return original_reset_idx(env_ids)

    raw_env._reset_idx = capturing_reset_idx
    try:
        yield capture
    finally:
        raw_env._reset_idx = original_reset_idx


In [ ]:
def run_corrected_prefall_log_rollout(checkpoint_path, blind_joint_indices, forward_speed_mps,
                                       env_seed=ENV_SEED, steps=250, window=PREFALL_LOG_WINDOW):
    """run_extended_prefall_log_rollout(セル80)と同じロールアウトに、
    capture_true_terminal_state によるリセット前スナップショット取得を追加した、
    訂正版。history の末尾は、従来通りの(転倒ステップでは汚染されている可能性のある)
    値のままにしておき、別途 'true_terminal' キーに、真の終端状態を格納する。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    robot = raw_env.scene["robot"]
    contact_sensor = raw_env.scene["feet_ground_contact"]

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()

    history = []
    fell_over = False
    fell_step = None
    true_terminal = None

    with capture_true_terminal_state(raw_env, robot, contact_sensor, env_idx=0) as capture:
        for step in range(steps):
            raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
            if blind_joint_indices:
                obs = blind_observation(obs, blind_joint_indices)
            with torch.no_grad():
                action = policy(obs)

            obs, _, dones, _ = env.step(action)

            # 従来通りの(終端ステップでは汚染されている可能性がある)記録
            record = {
                "t": step,
                "action": action[0].detach().cpu().numpy().copy(),
                "joint_pos": robot.data.joint_pos[0].detach().cpu().numpy().copy(),
                "joint_vel": robot.data.joint_vel[0].detach().cpu().numpy().copy(),
                "joint_pos_target": robot.data.joint_pos_target[0].detach().cpu().numpy().copy(),
                "joint_effort_target": robot.data.joint_effort_target[0].detach().cpu().numpy().copy(),
                "root_lin_vel_w": robot.data.root_link_lin_vel_w[0].detach().cpu().numpy().copy(),
                "foot_contact_found": contact_sensor.data.found[0].detach().cpu().numpy().copy(),
                "foot_contact_force": contact_sensor.data.force[0].detach().cpu().numpy().copy(),
            }
            history.append(record)
            if len(history) > window:
                history.pop(0)

            fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
            if fell:
                fell_over = True
                fell_step = step
                if capture["captured"]:
                    true_terminal = dict(capture)
                break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()
    return {"fell": fell_over, "fell_step": fell_step, "history": history, "true_terminal": true_terminal}


In [ ]:
import pickle

CORRECTED_LOG_DIR = "prefall_logs_corrected"
os.makedirs(CORRECTED_LOG_DIR, exist_ok=True)

# §3.6本文の集計(3件の転倒インスタンス)と、Figure 4の代表例の両方をカバーするため、
# §13と同じ条件・seed(MECH_SEEDS = [1,2,3,4])で再測定する。
for cond_name, joint_idx in [
    ("right_half+trunk", [JOINT_NAMES.index(n) for n in _cond_16r]),
    ("left_half+trunk", [JOINT_NAMES.index(n) for n in _cond_16l]),
]:
    for seed in MECH_SEEDS:
        out_path = os.path.join(CORRECTED_LOG_DIR, f"{cond_name.replace('+', '_')}_seed{seed}.pkl")
        if os.path.exists(out_path):
            continue
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
        result = run_corrected_prefall_log_rollout(ckpt, joint_idx, PREFALL_LOG_SPEED)
        with open(out_path, "wb") as f:
            pickle.dump(result, f)
        status = "転倒" if result["fell"] else "転倒せず"
        captured = "真の終端状態を取得済み" if result["true_terminal"] is not None else "(転倒なし、または未取得)"
        print(f"{cond_name} / seed{seed}: {status} (fell_step={result['fell_step']}) {captured if result['fell'] else ''}")

print("done:", CORRECTED_LOG_DIR)


### 27.1 §3.6本文の集計値を、修正後データで再計算

§13(セル43)と同じ「転倒した試行について、全29関節の終端角度とデフォルト角度の差」の
集計を、修正後の`true_terminal`から計算し直す。査読者への回答・本文で使用した
「ablated関節で約25°、intact関節で約16°」という数値が、この再測定でも再現されることを
確認する。

In [ ]:
log_files_corrected = sorted(glob.glob(os.path.join(CORRECTED_LOG_DIR, "*.pkl")))
print(f"{len(log_files_corrected)} 件のログファイルを検出。")

summary_rows = []
for path in log_files_corrected:
    with open(path, "rb") as f:
        result = pickle.load(f)
    if not result["fell"] or result["true_terminal"] is None:
        continue

    cond_name = "right_half+trunk" if "right_half" in os.path.basename(path) else "left_half+trunk"
    blinded_set = set(_cond_16r if cond_name == "right_half+trunk" else _cond_16l)

    terminal_joint_pos_deg = np.degrees(result["true_terminal"]["joint_pos"])
    diffs = []
    for i, name in enumerate(JOINT_NAMES):
        diff = terminal_joint_pos_deg[i] - default_joint_pos_all[i]
        diffs.append((name, name in blinded_set, diff))
    diffs_df = pd.DataFrame(diffs, columns=["joint", "was_blinded", "diff_from_default_deg"])
    diffs_df["abs_diff"] = diffs_df["diff_from_default_deg"].abs()

    blinded_mean_abs = diffs_df[diffs_df["was_blinded"]]["abs_diff"].mean()
    intact_mean_abs = diffs_df[~diffs_df["was_blinded"]]["abs_diff"].mean()
    min_abs = diffs_df["abs_diff"].min()

    print(f"{os.path.basename(path)}:")
    print(f"  感覚を失った関節群の、デフォルトとの平均絶対差: {blinded_mean_abs:.1f}度")
    print(f"  感覚が健在な関節群の、デフォルトとの平均絶対差: {intact_mean_abs:.1f}度")
    print(f"  全29関節中、デフォルトとの差が最小の関節: {min_abs:.2f}度")
    print()
    summary_rows.append({"file": os.path.basename(path), "blinded_mean_abs_deg": blinded_mean_abs,
                          "intact_mean_abs_deg": intact_mean_abs, "min_abs_deg": min_abs})

summary_df = pd.DataFrame(summary_rows)
print("=== 3件の転倒インスタンス全体の平均(本文記載値と比較) ===")
print(f"ablated関節群の平均絶対差: {summary_df['blinded_mean_abs_deg'].mean():.1f}度 (本文記載: 約25度)")
print(f"intact関節群の平均絶対差:  {summary_df['intact_mean_abs_deg'].mean():.1f}度 (本文記載: 約16度)")
print(f"全インスタンス中の最小絶対差: {summary_df['min_abs_deg'].min():.2f}度 (本文記載: 0.5度を下回るものはない)")


### 27.2 正常歩行時のベースライン逸脱(約3度)の確認

§52(`g1_sensory_screening_metrics.csv`、baseline行、ablationなし)から、正常歩行時の
関節角度のdefaultからの平均絶対偏差を算出し、本文の「約3°」という記載を確認する。

In [ ]:
if os.path.exists(METRICS_CSV):
    _metrics_df = pd.read_csv(METRICS_CSV)
    _baseline_rows = _metrics_df[_metrics_df["joint_name"] == "(baseline)"]
    # joint_tracking_rms_deg は「平均二乗偏差の平方根」(RMS)。本文の「平均絶対偏差」に
    # 近い値として参照する(厳密な単純平均絶対偏差を求める場合は、生ロールアウトの
    # 再測定が必要)。
    print("正常歩行(ablationなし)の関節角度RMS偏差(度):")
    print(_baseline_rows[["seed", "joint_tracking_rms_deg"]].to_string(index=False))
    print(f"\n平均: {_baseline_rows['joint_tracking_rms_deg'].mean():.1f}度 (本文記載: 約3度)")
else:
    print(f"{METRICS_CSV} が見つかりません。§16(セル51以降)を先に実行してください。")


### 27.3 Figure 4の代表例(right_half+trunk, seed 1, 0.5 m/s)の、修正後の数値を算出

従来の§26(セル81)の`tracking_error_deg`は、転倒ステップの`joint_pos`がリセット後の
デフォルト値で汚染されていたため、不正確だった。`true_terminal`から計算し直す。

In [ ]:
REP_PATH = os.path.join(CORRECTED_LOG_DIR, "right_half_trunk_seed1.pkl")
with open(REP_PATH, "rb") as f:
    rep_result = pickle.load(f)

assert rep_result["fell"] and rep_result["true_terminal"] is not None, \
    "代表例(seed1)が転倒していない、または真の終端状態が未取得です。PREFALL_LOG_SEEDSにseed1が含まれているか確認してください。"

tt = rep_result["true_terminal"]
tracking_error_deg_corrected = np.degrees(np.abs(tt["joint_pos"] - tt["joint_pos_target"]))

print(f"fell_step = {rep_result['fell_step']}")
print(f"[修正後] 関節追従誤差(joint_pos vs joint_pos_target)の最大値: "
      f"{tracking_error_deg_corrected.max():.1f}度 "
      f"(最大の関節: {JOINT_NAMES[int(np.argmax(tracking_error_deg_corrected))]})")
print(f"[修正後] 転倒直前の鉛直方向ルート速度: {tt['root_lin_vel_w'][2]:.2f} m/s")
print(f"[修正後] 転倒時の足接地点数(左右): {tt['foot_contact_found']}")
print()
print("(参考)従来の§26集計(リセット汚染の影響を受けている可能性がある値):")
last_old = rep_result["history"][-1]
tracking_error_deg_old = np.degrees(np.abs(last_old["joint_pos"] - last_old["joint_pos_target"]))
print(f"  従来方式での最大値: {tracking_error_deg_old.max():.1f}度")


### 27.4 Figure 4の再生成(right_half+trunk, seed 1, 0.5 m/s、最終50ステップ)

`history`(直近`window`ステップ)の時系列と、転倒ステップのみ`true_terminal`で置き換えた
系列を使って、ルート鉛直速度・右膝(ablated)/左膝(intact)の関節角度・足接地点数のプロットを
再生成する。

In [ ]:
import matplotlib.pyplot as plt

right_knee_idx = JOINT_NAMES.index("right_knee")
left_knee_idx = JOINT_NAMES.index("left_knee")

hist = rep_result["history"]
ts = [r["t"] for r in hist]
root_vz = [r["root_lin_vel_w"][2] for r in hist]
right_knee_deg = [np.degrees(r["joint_pos"][right_knee_idx]) for r in hist]
left_knee_deg = [np.degrees(r["joint_pos"][left_knee_idx]) for r in hist]
foot_contact_n = [int(np.sum(r["foot_contact_found"])) for r in hist]

# 転倒ステップ(最終ステップ)だけ、リセット汚染のない真の終端値で置き換える
if tt is not None:
    root_vz[-1] = tt["root_lin_vel_w"][2]
    right_knee_deg[-1] = np.degrees(tt["joint_pos"][right_knee_idx])
    left_knee_deg[-1] = np.degrees(tt["joint_pos"][left_knee_idx])
    foot_contact_n[-1] = int(np.sum(tt["foot_contact_found"]))

fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

axes[0].plot(ts, root_vz, color="tab:red")
axes[0].set_ylabel("Root vertical\nvelocity (m/s)")
axes[0].axhline(0, color="gray", linewidth=0.5)

axes[1].plot(ts, right_knee_deg, label="right_knee (ablated)", color="tab:orange")
axes[1].plot(ts, left_knee_deg, label="left_knee (intact)", color="tab:blue")
axes[1].axhline(np.degrees(default_joint_pos_all[right_knee_idx]), color="tab:orange", linestyle=":", linewidth=1, label="right_knee default")
axes[1].axhline(np.degrees(default_joint_pos_all[left_knee_idx]), color="tab:blue", linestyle=":", linewidth=1, label="left_knee default")
axes[1].set_ylabel("Joint angle (deg)")
axes[1].legend(fontsize=7, loc="upper left")

axes[2].step(ts, foot_contact_n, where="post", color="tab:green")
axes[2].set_ylabel("Feet in contact\n(count)")
axes[2].set_xlabel("Simulation step (final window before fall)")
axes[2].set_yticks([0, 1, 2])

fig.suptitle("Corrected: right_half+trunk, policy seed 1, 0.5 m/s\n(joint angles diverge from, not converge to, default)")
fig.tight_layout()
fig.savefig("figure4_corrected.png", dpi=200)
plt.show()

print("保存しました: figure4_corrected.png")
print("この図が、§3.6訂正後のFigure 4の差し替え用データです。")
print("(参考として、各関節のdefault角度を破線で重ねてあり、膝関節がdefaultへ")
print(" 収束するのではなく、むしろdefaultから乖離していく様子を示しています。)")
